In [37]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "manrique2010great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2a_of_JEP.csv")
complete_path_2 = os.path.join(original_data_pathway, "2b_of_JEP.csv")

complete_path_3 = os.path.join(original_data_pathway, "exp_1.csv")
complete_path_4 = os.path.join(original_data_pathway, "exp_3.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [38]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1['experiment_name']='2a'
df1.rename(columns={"Date": "date_temp"}, inplace=True)
# df1['date_temp'] = df1['date_temp'].astype(str)
temp_var = ''
out_list = []
for index, row in df1.iterrows():
    if not pd.isna(row['date_temp']):
        temp_var = row['date_temp']
    out_list.append(temp_var)
df1 = df1.assign(date=out_list)
df1[['day','month', 'year']] = df1['date'].str.split('/',expand=True)
df1['year'] = '20' + df1['year'].astype(str)




df2 = pd.read_csv(complete_path_2)
df2['experiment_name']='2b'


In [39]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "participant_temp",
        "species":"species_original"}, inplace=True)
    x['participant_temp'] = x['participant_temp'].str.rstrip()
    x['study_id']="manrique2010great"
    x['experiment']="2"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [40]:
temp_var = ''
out_list = []
for index, row in fulldf.iterrows():
    if not pd.isna(row['participant_temp']):
        temp_var = row['participant_temp']
    out_list.append(temp_var)
fulldf = fulldf.assign(participant=out_list)

In [41]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [42]:
fulldf.columns = fulldf.columns.str.replace(' ', '_')
# fulldf.columns

fulldf.loc[fulldf.experiment_name == '2b', ['condition']] = 'visual_static'

In [43]:
exp_2b_dates = [['dunja',2009, 8, 20],
                ['padana',2009, 8, 20],
                ['kila',2009, 8, 20],
                ['joey',2009, 8, 21],
                ['yasa',2009, 8, 21],
                ['limbuko',2009, 8, 21],
                ['kuno',2009, 8, 21],
                ['dokana',2009, 8, 22],
                ['pini',2009, 8, 22],
                ['jahaga',2009, 9, 10],
                ['gertrudia',2009, 9, 10],
                ['fraukje',2009, 9, 12],
                ['frodo',2009, 9, 12],
                ['patrick',2009, 9, 12],
                ['ulla',2009, 9, 12]]
for index, row in fulldf.iterrows():
    if row['condition'] == 'visual_static':
        for x,y,m,d in exp_2b_dates:
            fulldf.loc[fulldf.participant == x, ['year', 'month','day']] = y,m,d



In [44]:
choice_rename = [['r', 'rigid'],
                 ['f','flexible']]
for x,y in choice_rename:
    fulldf['apes_choice'].replace(x, y, inplace=True)

In [45]:
fulldf=fulldf.sort_values(by = ['participant', 'month', 'day'])
output_no_2 = 1 
repeat_2 = 0 
temp_var_2 = []
for index, row in fulldf.iterrows():
    if row['experiment_name'] == '2a': 
        temp_var_2.append(output_no_2) 
        repeat_2 = repeat_2+1 
        if repeat_2 == 6: 
            output_no_2 = output_no_2+1 
            repeat_2 = 0
        if output_no_2 == 3: 
            output_no_2 = 1 
    elif row['experiment_name'] == '2b':
         temp_var_2.append(1)
fulldf = fulldf.assign(session=temp_var_2)

In [46]:
fulldf=fulldf.sort_values(by = ['participant', 'condition'])
temp_var = 1
out_list_3 = []
for index, row in fulldf.iterrows():
    out_list_3.append(temp_var)
    temp_var = temp_var+1
    if temp_var == 7: 
        temp_var = 1
fulldf = fulldf.assign(trial=out_list_3)


In [47]:
combine_c_list = [['yes','rigid'],
                    ['no','flexible']]
for x,y in combine_c_list:
    fulldf.loc[fulldf.correct == x, ['apes_choice']] = y
    fulldf.loc[fulldf.apes_choice == y, ['correct']] = x



In [48]:
fulldf.rename(columns={'tool-set': "tool-set_exp-2a",
                       'tool_set_administered': "tool-set_exp-2b"}, inplace=True)

fulldf['experiment']='2'

In [49]:
df3 = pd.read_csv(complete_path_3)
df3['experiment']='1'



In [50]:
df4 = pd.read_csv(complete_path_4)
df4['experiment']='3'

# remdf=["frodo", "bimbo", "ulla",'viringika']
# df4 = df4[~df4.participant.isin(re


In [51]:

fulldf_temp = pd.concat([df3,df4], ignore_index=True, sort=False)

fulldf_temp['study_id']="manrique2010great"   
fulldf_temp= fulldf_temp.merge(apedf,left_on='participant', right_on='name', how='left')
fulldf_temp['year'].unique()

replace_list = ['year','month','day']
for x in replace_list:
    fulldf_temp[x].replace(np.nan, '99999', inplace=True, regex=True)




In [52]:


fulldf_temp['year']=fulldf_temp['year'].astype(int)
fulldf_temp['month']=fulldf_temp['month'].astype(int)
fulldf_temp['day']=fulldf_temp['day'].astype(int)

fulldf_new = pd.concat([fulldf,fulldf_temp], ignore_index=True, sort=False)

In [53]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf_new= fulldf_new.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf_new['dodc'] = fulldf_new['year'].astype(str) + '-' + fulldf_new['month'].astype(str) + '-' + fulldf_new['day'].astype(str)
fulldf_new['dodc'].replace('99999-99999-99999', np.nan, inplace=True, regex=True)
fulldf_new['dodc'] = pd.to_datetime(fulldf_new['dodc'])
fulldf_new['dob'] = pd.to_datetime(fulldf_new['dob'])

fulldf_new['age_in_years'] = (fulldf_new['dodc'] - fulldf_new['dob']).dt.days//365

# fulldf_new.columns

replace_list = ['year','month','day']
for x in replace_list:
    fulldf_new[x].replace( 99999,np.nan, inplace=True, regex=True)

In [54]:

fulldf_new=fulldf_new[['study_id','experiment','experiment_name','year', 'month','day',  
        'participant','age_original','age_in_years','sex','species', 'session',
        'trial', 'condition', 'tool-set_exp-2a','tool-set_exp-2b', 'correct', 'try',  'apes_choice',
         'tool_set', 'transport', 'transport_tools', 'use',
       'use_tools', 'present', 'comments', 'drop_out' ]]

# comp_out_path_stand = os.path.join(out_pathway, 'manrique2010great_exp2_standardized.csv')
# fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

# names = fulldf.columns.tolist()
# df = pd.DataFrame(names)
# df = df.rename(columns={0: "column_name"})
# df["description"] = ""
# studyID_glossary=df[["column_name", "description"]]

# comp_out_path_glossary = os.path.join(out_pathway, 'manrique2010great_exp2_glossary.csv')
# studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

for index in range(1,4):
    exp = fulldf_new[fulldf_new['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'manrique2010great_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'manrique2010great_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

